In [1]:
# Importar

import os
import time
import pandas as pd
from groq import Groq
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv()

# Cliente Groq — mesmo que usaste no P2
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.3-70b-versatile"  # modelo com contexto grande

print("✅ Imports prontos")
print(f"Modelo: {MODEL}")

✅ Imports prontos
Modelo: llama-3.3-70b-versatile


In [2]:
# Carregar o dataset
# Usamos o HaluEval: tem respostas com hallucinations já identificadas
# Perfeito para testar se o juiz as detecta correctamente

from datasets import load_dataset

ds = load_dataset(
    "pminervini/HaluEval",
    "qa_samples",
    split="data",
    streaming=True
)

# Pegar 20 exemplos
amostras = [next(iter(ds)) for _ in range(20)]
df = pd.DataFrame(amostras)

print(f"✅ {len(df)} exemplos carregados")
print(f"Colunas: {df.columns.tolist()}")
df.head(3)

✅ 20 exemplos carregados
Colunas: ['knowledge', 'question', 'answer', 'hallucination']


,knowledge,question,answer,hallucination
0,Arthur's Magazine (1844–1846) was an American ...,Which magazine was started first Arthur's Maga...,First for Women was started first.,yes
1,Arthur's Magazine (1844–1846) was an American ...,Which magazine was started first Arthur's Maga...,First for Women was started first.,yes
2,Arthur's Magazine (1844–1846) was an American ...,Which magazine was started first Arthur's Maga...,First for Women was started first.,yes


In [4]:
# Ver um exemplo
# Entender o que precisa ser analisado

exemplo = df.iloc[0]

print("PERGUNTA:")
print(exemplo["question"])
print("\nRESPOSTA COM HALLUCINATION:")
print(exemplo["hallucination"])
print("\nRESPOSTA CORRECTA:")
print(exemplo["answer"])

PERGUNTA:
Which magazine was started first Arthur's Magazine or First for Women?

RESPOSTA COM HALLUCINATION:
yes

RESPOSTA CORRECTA:
First for Women was started first.


In [5]:
# LLM-as-a-Judge
# O juiz recebe pergunta + resposta + referência
# e classifica o tipo de hallucination

def llm_judge(question, answer, reference):
    prompt = f"""You are an expert evaluator specialized in detecting
hallucinations in AI-generated responses.

Question: {question[:300]}
Answer to evaluate: {answer[:400]}
Reference (correct answer): {reference[:400]}

Classify the answer into exactly one category:
- CORRECT: factually aligned with the reference
- FACTUAL_ERROR: contains a verifiable factual mistake
- FABRICATION: invents information not present in the reference
- CONTRADICTION: directly contradicts the reference

Reply in this exact format:
CATEGORY | confidence (0.0-1.0) | one line explanation"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        temperature=0  # determinístico — sempre a mesma resposta
    )

    return response.choices[0].message.content.strip()


# Teste rápido com 1 exemplo
resultado = llm_judge(
    question=df.iloc[0]["question"],
    answer=df.iloc[0]["hallucination"],
    reference=df.iloc[0]["answer"]
)

print("Teste do juiz:")
print(resultado)
print("✅ Função pronta" if "|" in resultado else "⚠️ Formato inesperado")

Teste do juiz:
FACTUAL_ERROR | 1.0 | The answer "yes" does not provide a clear comparison between the two magazines and does not align with the reference that states First for Women was started first.
✅ Função pronta


In [6]:
# Colocar em todos os elementos
# Pausa entre chamadas para não exceder rate limit do Groq

resultados = []

for i, row in df.iterrows():
    print(f"A avaliar exemplo {i+1}/20...", end=" ")

    try:
        verdict = llm_judge(
            question=row["question"],
            answer=row["hallucination"],
            reference=row["answer"]
        )

        # Separar os 3 campos do resultado
        partes = verdict.split("|")
        categoria  = partes[0].strip() if len(partes) > 0 else "UNKNOWN"
        confianca  = partes[1].strip() if len(partes) > 1 else "0.0"
        explicacao = partes[2].strip() if len(partes) > 2 else ""

        resultados.append({
            "question":    row["question"][:100],
            "answer":      row["hallucination"][:100],
            "reference":   row["answer"][:100],
            "categoria":   categoria,
            "confianca":   confianca,
            "explicacao":  explicacao
        })

        print(f"→ {categoria}")

    except Exception as e:
        print(f"→ erro: {str(e)[:50]}")
        resultados.append({
            "question": row["question"][:100],
            "categoria": "ERROR",
            "confianca": "0.0",
            "explicacao": str(e)[:100]
        })

    time.sleep(1)  # pausa para evitar rate limit

df_resultados = pd.DataFrame(resultados)
print("\n✅ Avaliação completa!")

A avaliar exemplo 1/20... → FACTUAL_ERROR
A avaliar exemplo 2/20... → FACTUAL_ERROR
A avaliar exemplo 3/20... → FACTUAL_ERROR
A avaliar exemplo 4/20... → FACTUAL_ERROR
A avaliar exemplo 5/20... → FACTUAL_ERROR
A avaliar exemplo 6/20... → FACTUAL_ERROR
A avaliar exemplo 7/20... → FACTUAL_ERROR
A avaliar exemplo 8/20... → FACTUAL_ERROR
A avaliar exemplo 9/20... → FACTUAL_ERROR
A avaliar exemplo 10/20... → FACTUAL_ERROR
A avaliar exemplo 11/20... → FACTUAL_ERROR
A avaliar exemplo 12/20... → FACTUAL_ERROR
A avaliar exemplo 13/20... → FACTUAL_ERROR
A avaliar exemplo 14/20... → FACTUAL_ERROR
A avaliar exemplo 15/20... → FACTUAL_ERROR
A avaliar exemplo 16/20... → FACTUAL_ERROR
A avaliar exemplo 17/20... → FACTUAL_ERROR
A avaliar exemplo 18/20... → FACTUAL_ERROR
A avaliar exemplo 19/20... → FACTUAL_ERROR
A avaliar exemplo 20/20... → FACTUAL_ERROR

✅ Avaliação completa!


In [7]:
# Análise

print("DISTRIBUIÇÃO DE CATEGORIAS:")
print("=" * 40)
print(df_resultados["categoria"].value_counts())

print("\nEXEMPLOS POR CATEGORIA:")
print("=" * 40)

for categoria in df_resultados["categoria"].unique():
    subset = df_resultados[df_resultados["categoria"] == categoria]
    print(f"\n{categoria} ({len(subset)} casos):")
    for _, row in subset.head(2).iterrows():
        print(f"  → {row['explicacao']}")

DISTRIBUIÇÃO DE CATEGORIAS:
categoria
FACTUAL_ERROR    20
Name: count, dtype: int64

EXEMPLOS POR CATEGORIA:

FACTUAL_ERROR (20 casos):
  → The answer "yes" does not provide a clear comparison between the two magazines and does not specify which one was started first, contradicting the reference that clearly states First for Women was started first.
  → The answer "yes" does not provide a clear comparison between the two magazines and does not align with the reference that states First for Women was started first.


In [8]:
# Guardar

import os
os.makedirs("../results", exist_ok=True)

df_resultados.to_csv("../results/llm_judge_results.csv", index=False)

print("RESUMO FINAL:")
print("=" * 40)
print(f"Total avaliados:  {len(df_resultados)}")
print(f"Correctos:        {(df_resultados['categoria'] == 'CORRECT').sum()}")
print(f"Factual errors:   {(df_resultados['categoria'] == 'FACTUAL_ERROR').sum()}")
print(f"Fabrications:     {(df_resultados['categoria'] == 'FABRICATION').sum()}")
print(f"Contradictions:   {(df_resultados['categoria'] == 'CONTRADICTION').sum()}")
print(f"Erros de API:     {(df_resultados['categoria'] == 'ERROR').sum()}")

print("\n✅ Guardado em results/llm_judge_results.csv")

RESUMO FINAL:
Total avaliados:  20
Correctos:        0
Factual errors:   20
Fabrications:     0
Contradictions:   0
Erros de API:     0

✅ Guardado em results/llm_judge_results.csv
